In [ ]:
import sys; sys.path.insert(0, "..")
from fantasy.yahoo_client import get_my_roster, get_current_matchup, get_roster_by_team, get_league_categories
from fantasy.nba_stats import get_player_stats, get_games_this_week
from fantasy.analysis import project_team_categories, classify_categories
from fantasy.llm import build_matchup_prompt, ask_gemini
import pandas as pd

In [ ]:
matchup = get_current_matchup()
my_roster = get_my_roster()
opp_roster = get_roster_by_team(matchup["opponent_team_key"])
categories = get_league_categories()
week_start, week_end = matchup["week_start"], matchup["week_end"]
print(f"Week {matchup['week']}: {week_start} → {week_end}")
print(f"Opponent team: {matchup['opponent_team_key']}")

In [ ]:
def enrich_players(roster):
    enriched = []
    for p in roster:
        stats = get_player_stats(p["name"])
        games = get_games_this_week(p["team_abbr"], week_start, week_end)
        enriched.append({**p, "stats": stats, "games_remaining": games})
    return enriched

my_players = enrich_players(my_roster)
opp_players = enrich_players(opp_roster)

In [ ]:
my_proj = project_team_categories(my_players)
opp_proj = project_team_categories(opp_players)
classification = classify_categories(my_proj, opp_proj)

df = pd.DataFrame({
    "My Team": my_proj,
    "Opponent": opp_proj,
    "Status": classification,
}).round(1)
print(df.to_string())

In [ ]:
prompt = build_matchup_prompt(my_proj, opp_proj, classification)
advice = ask_gemini(prompt)
print("\n=== WEEKLY GAME PLAN ===\n")
print(advice)